# SeeSR evaluation on Google Colab: reproduction, few-step comparison and method ablations

Two separate tracks per dataset, never mixed:

* **Track A, reproduction:** `baseline_ddpm50`, the original SeeSR configuration with upstream RNG behaviour,
  compared only with the published numbers.
* **Track B, controlled comparison:** `{variant}_ddpm{50,30,20}_s{seed}`, identical settings with per-image
  seeding. Variants: `original` (upstream SeeSR), `semantic_conf` (component A), `lr_schedule` (component B),
  `combined` (A + B), optional `lr_schedule_late`. Only the method, the step count and the seed differ;
  results are paired by (image, seed). Method description: `cfa_seesr/README.md`.
* **Spatial semantic reliability (SSR)**, an optional extension of `semantic_conf`, runs only if the Gate 0
  attention check (section 8b) says GO and the 40-image screening (section 15) says CONTINUE.

Everything (images, metadata, logs, evaluation CSV/JSON, comparisons) is written to Google Drive. Re-running a cell
after a disconnect skips completed runs and stops on incomplete ones.

**Before running**
1. *Runtime → Change runtime type →* **L4** (or A100) GPU.
2. Commit and push `experiments/` and `cfa_seesr/` to `research/phase2-baseline` (Colab clones it from GitHub).
3. Put weights and datasets in Drive (sections 4 and 7). Nothing is downloaded automatically.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv

## 2. Configuration (edit this cell only)

In [ ]:
import os, json, pathlib, subprocess
REPO_URL   = "https://github.com/rithvik318/SeeSR.git"
BRANCH     = "research/phase2-baseline"
REPO_DIR   = "/content/SeeSR"
DRIVE_ROOT = "/content/drive/MyDrive/seesr"
MODELS_DIR = f"{DRIVE_ROOT}/models"
DATA_DIR   = f"{DRIVE_ROOT}/datasets"
RESULTS    = f"{DRIVE_ROOT}/results"          # <RESULTS>/<dataset>/<experiment>/...
ENV_DIR    = "/content/seesr-env"
PY         = f"{ENV_DIR}/bin/python"

# Point these at the folders you uploaded; validation (section 7) reports what it actually finds.
DATASETS = {
    "RealSR":    {"lr": f"{DATA_DIR}/RealSR/test_LR",    "gt": f"{DATA_DIR}/RealSR/test_HR",  "strip": []},
    "DRealSR":   {"lr": f"{DATA_DIR}/DRealSR/test_LR",   "gt": f"{DATA_DIR}/DRealSR/test_HR", "strip": []},
    "RealLR200": {"lr": f"{DATA_DIR}/RealLR200",         "gt": None,                          "strip": []},
}
TRACK_A = "baseline_ddpm50"
STEPS = [50, 30, 20]
SEEDS = [42]                     # start with one seed; later set [42, 123, 2026] and re-run: finished runs are skipped
VARIANTS = ["original", "semantic_conf", "lr_schedule", "combined"]   # + "lr_schedule_late" (optional ablation)
# Spatial semantic reliability (SSR): only after Gate 0 = GO (section 8b) and a passed screening (section 15)
SSR_VARIANTS = ["spatial_sem", "semantic_spatial", "combined_spatial", "spatial_shuffled", "global_meanmatched"]
SCREEN_VARIANTS = ["original", "semantic_conf", "spatial_sem", "semantic_spatial", "spatial_shuffled"]
SCREEN_STEPS, SCREEN_SEED, SCREEN_N = [20, 50], 42, 20
GATE0 = SCREEN = "NOT RUN"      # set by sections 8b and 15; anything but GO / CONTINUE skips SSR
def track_b(seeds=None, variants=None):
    return [f"{v}_ddpm{st}_s{sd}" for v in (variants or VARIANTS) for st in STEPS for sd in (seeds or SEEDS)]

os.environ.update(PY=PY, PYTHON=PY, LOG_DIR=f"{RESULTS}/logs",
                  PYTHONDONTWRITEBYTECODE="1")   # upstream tracks __pycache__/*.pyc in git

def sh(cmd):
    """Run a bash command, stream its output, stop the notebook on failure.

    Uses subprocess rather than `!`/get_ipython().system(), which would re-expand {...} and $... in cmd.
    """
    proc = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise RuntimeError(f"exit code {proc.returncode}: {cmd[:300]}")

## 3. Drive and code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{RESULTS}/logs", exist_ok=True)
if not os.path.isdir(REPO_DIR):
    sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
sh("git log -1 --format='%H %s' && git status --short")
assert pathlib.Path("experiments/run_inference.py").exists(), "experiments/ not on this branch: commit and push it first"

## 4. Python 3.10 environment (pinned; ≈ 5 min)

In [ ]:
sh("pip -q install uv")
sh("uv python install 3.10")
if not os.path.exists(PY):
    sh(f"uv venv --seed --python 3.10 {ENV_DIR}")
# pip (not uv) so the PyTorch cu118 index and PyPI are merged for torch / xformers
sh(f"{PY} -m pip -q install -r experiments/requirements-colab.txt -r experiments/requirements-metrics.txt pytest")

## 5. Verify the stack and snapshot the environment to Drive

In [ ]:
sh(f'''{PY} - <<'EOF'
import sys, torch, diffusers, transformers, huggingface_hub, pyiqa
print("python", sys.version.split()[0], "| torch", torch.__version__, "cuda", torch.version.cuda)
assert torch.cuda.is_available(), "no GPU visible to the venv"
p = torch.cuda.get_device_properties(0)
print("gpu", p.name, round(p.total_memory / 2**30, 1), "GiB, sm", f"{{p.major}}{{p.minor}}")
x = torch.randn(1024, 1024, device="cuda", dtype=torch.float16); (x @ x).sum().item()
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__,
      "| hub", huggingface_hub.__version__, "| pyiqa", pyiqa.__version__)
try:
    import xformers, xformers.ops as xo
    q = torch.randn(1, 64, 8, 40, device="cuda", dtype=torch.float16); xo.memory_efficient_attention(q, q, q)
    print("xformers", xformers.__version__, "kernels OK")
except Exception as e:
    print("xformers NOT usable -> append --set runtime.enable_xformers=auto to every run:", type(e).__name__, e)
sys.path.insert(0, "."); import test_seesr; print("upstream test_seesr imports OK")
EOF''')
env_dir = f"{RESULTS}/env"; os.makedirs(env_dir, exist_ok=True)
sh(f"{PY} -m pip freeze > {env_dir}/pip_freeze.txt && nvidia-smi > {env_dir}/nvidia_smi.txt && git rev-parse HEAD > {env_dir}/git_commit.txt")
sh(f"{PY} -m pytest -q -p no:cacheprovider experiments/tests")

## 6. Model weights (manual, one-time) and validation

Put these in `MODELS_DIR` on Drive (sources: upstream README):

| Path under `MODELS_DIR` | Contents |
|---|---|
| `stable-diffusion-2-base/` | `scheduler/`, `text_encoder/`, `tokenizer/`, `vae/`, `feature_extractor/` (diffusers layout) |
| `seesr/` | `unet/`, `controlnet/` from the SeeSR release |
| `DAPE.pth` | SeeSR release |
| `ram_swin_large_14m.pth` | RAM release |

`preset/models` is symlinked to `MODELS_DIR` because the upstream code hard-codes that path. The check below
verifies required files, git-lfs pointer files, `use_image_cross_attention` in the SeeSR configs, the scheduler
`prediction_type`, and prints the DDPM timesteps the real scheduler config produces for 50 / 30 / 20 steps.

In [ ]:
preset = pathlib.Path(REPO_DIR, "preset", "models")
if preset.is_symlink():
    preset.unlink()
if preset.exists():
    raise SystemExit(f"{preset} exists and is not a symlink; move it aside first")
preset.symlink_to(MODELS_DIR, target_is_directory=True)
sh(f"{PY} experiments/tools/validate_inputs.py checkpoints --config experiments/configs/baseline.yaml --output {RESULTS}/env/checkpoints_report.json")

## 7. Dataset validation

SeeSR's protocol: RealSR (100 pairs) and DRealSR (93 pairs) as distributed with the StableSR test sets, with LR
128×128 center crops and HR 512×512; RealLR200 is 200 LR images without GT (no-reference metrics only).
Images are **not** modified. The check fails on missing or extra files, duplicate names or content,
non-image files, and pairs where HR ≠ 4 × LR. Fix the folders, not the images.

In [ ]:
for ds, d in DATASETS.items():
    if not os.path.isdir(d["lr"]):
        print(f"skip {ds}: {d['lr']} not found"); continue
    cmd = f"{PY} experiments/tools/validate_inputs.py dataset --name {ds} --lr_dir {d['lr']} --scale 4 --output {RESULTS}/{ds}/dataset_report.json"
    if d["gt"]:
        cmd += f" --gt_dir {d['gt']}"
    cmd += "".join(f" --strip_suffix {s}" for s in d["strip"])
    print(f"===== {ds}"); sh(cmd)

## 8. Smoke test (one image, debugging only)

Uses `preset/datasets/test_datasets/179.png` from the repository. Results go to `<RESULTS>/_debug/` and are
**not benchmark results**. The second command runs the `combined` variant on the same image to check that the
method hooks load with the real models (the semantic record and the logged LR factors are printed).

In [ ]:
sh(f"{PY} experiments/run_inference.py --config experiments/configs/baseline.yaml --name smoke_ddpm50 "
   f"--set dataset.name=upstream_sample_DEBUG --set dataset.lr_dir=preset/datasets/test_datasets --set dataset.gt_dir=null "
   f"--results_root {RESULTS}/_debug --overwrite")
m = json.loads(pathlib.Path(RESULTS, "_debug", "smoke_ddpm50", "run_metadata.json").read_text())
print(json.dumps({k: m[k] for k in ("status", "attention_backend", "runtime_verified", "timing")}, indent=1))
print("executed denoising steps per image:", m["scheduler"]["denoising_steps_per_call"])

sh(f"{PY} experiments/run_inference.py --sweep experiments/configs/sampling_sweep.yaml --only combined_ddpm20_s42 "
   f"--set dataset.name=upstream_sample_DEBUG --set dataset.lr_dir=preset/datasets/test_datasets --set dataset.gt_dir=null "
   f"--results_root {RESULTS}/_debug --overwrite")
run = pathlib.Path(RESULTS, "_debug", "combined_ddpm20_s42")
m = json.loads((run / "run_metadata.json").read_text())
print(m["status"], json.dumps(m["method"], indent=1))
print((run / "semantic_confidence.jsonl").read_text()[:1500])

# SSR hooks with the real models (debug only; says nothing about whether SSR helps)
sh(f"{PY} experiments/run_inference.py --sweep experiments/configs/sampling_sweep.yaml --only combined_spatial_ddpm20_s42 "
   f"--set dataset.name=upstream_sample_DEBUG --set dataset.lr_dir=preset/datasets/test_datasets --set dataset.gt_dir=null "
   f"--results_root {RESULTS}/_debug --overwrite")
m = json.loads(pathlib.Path(RESULTS, "_debug", "combined_spatial_ddpm20_s42", "run_metadata.json").read_text())
print(m["status"], json.dumps(m["method"].get("spatial_summary"), indent=1))

## 8b. Gate 0: does DAPE's attention localise its tags? (decides whether SSR is used at all)

DAPE only, no diffusion: 50 LR images (25 RealSR + 25 DRealSR, chosen by filename hash). For every tag DAPE
predicts, the 20 % of grid cells it attends to most are occluded, and the logit drop is compared with occluding
the same cell pattern shifted to random other places. Tags of one image are not independent, so the test is over
images (bootstrap CI, sign-flip test). Protocol and limitations: `experiments/tools/gate0_attention.py`.
**GO** enables the SSR screening (section 15); **NO-GO** skips every SSR variant. The result is saved to Drive and
not recomputed unless `FORCE_GATE0 = True`.

In [ ]:
FORCE_GATE0 = False
G0 = pathlib.Path(RESULTS, "gate0")
if FORCE_GATE0 or not (G0 / "gate0_summary.json").exists():
    srcs = " ".join(f"--images {ds}={DATASETS[ds]['lr']}" for ds in ("RealSR", "DRealSR") if os.path.isdir(DATASETS[ds]["lr"]))
    sh(f"{PY} experiments/tools/gate0_attention.py {srcs} --n_images 50 --output_dir {G0}")
g0 = json.loads((G0 / "gate0_summary.json").read_text())
GATE0 = g0["decision"]
il = g0["image_level"]
print(json.dumps({"images_with_tags": g0["n_images_with_tags"], "tags": g0["n_tags"],
                  "mean_drop_top": g0["tag_level"]["drop_top"]["mean"], "mean_drop_control": g0["tag_level"]["drop_rand"]["mean"],
                  "image_level_mean_diff": il["mean_diff"]["mean"], "image_level_median_diff": il["mean_diff"]["median"],
                  "bootstrap_95ci": il["bootstrap_95ci_mean_diff"], "sign_flip_p": il["sign_flip_p_one_sided"],
                  "wilcoxon_p": il["wilcoxon_p_one_sided"], "fraction_images_positive": il["fraction_images_positive"],
                  "checks": g0["checks"]}, indent=1))
from IPython.display import HTML, Image as IPImage, display
colour = "#1a7f37" if GATE0 == "GO" else "#cf222e"
display(HTML(f"<h2 style='color:{colour}'>GATE 0: {GATE0}</h2><p>{g0['interpretation']}</p>"))
for f in sorted((G0 / "overlays").glob("*.png"))[:4]:
    display(IPImage(filename=str(f)))

### Optional: debugging subset of a real dataset

Symlinks the first `N` images (not copies, not modified) into `<RESULTS>/_debug/<ds>_DEBUG<N>`. Use for checking
the pipeline only; never report these numbers.

In [ ]:
def debug_subset(ds, n=5):
    d = DATASETS[ds]; dst = pathlib.Path(RESULTS, "_debug", f"{ds}_DEBUG{n}")
    lr = sorted(p for p in pathlib.Path(d["lr"]).iterdir() if p.suffix.lower() in (".png", ".jpg", ".jpeg"))[:n]
    (dst / "lr").mkdir(parents=True, exist_ok=True)
    for p in lr:
        link = dst / "lr" / p.name
        if not link.exists(): link.symlink_to(p)
    return str(dst / "lr")
# example: python-level call, then run_dataset-style commands with --set dataset.lr_dir=<returned path>
# print(debug_subset("RealSR", 5))

## 9. Runner

* A run is `completed` only if `test_seesr.main()` returned **and** every expected output image exists; its
  `run_metadata.json` is written last. A session that dies mid-run leaves `status: running` (or `incomplete`/`failed`),
  which `finished()` refuses to treat as done.
* `--overwrite` moves the old folder to `<name>__replaced_<time>` instead of writing into it, so outputs of two
  runs are never mixed. Evaluation only runs on completed runs.
* `tools/compare_results.py sampling` rejects the comparison (and lists every difference) if anything other than the
  method variant, the step count and the seed differs between track-B runs.
* Evaluation per run: paired (PSNR/SSIM/LPIPS/DISTS, GT datasets), no-reference (MUSIQ/CLIPIQA) and the
  LR-consistency diagnostic (bicubic-downsampled SR vs the LR input; a fidelity check, not a quality metric).

In [ ]:
def finished(ds, exp, root=None):
    f = pathlib.Path(root or f"{RESULTS}/{ds}", exp, "run_metadata.json")
    if not f.exists():
        return False
    status = json.loads(f.read_text())["status"]
    if status != "completed":
        raise RuntimeError(f"{ds}/{exp} has status '{status}'. Inspect it, then rerun that experiment with "
                           f"--overwrite (old outputs are moved aside, not mixed).")
    return True

def _sets(ds, extra="", data=None, root=None):
    d = data or DATASETS[ds]
    gt = d["gt"] if d["gt"] else "null"
    return (f"--set dataset.name={ds} --set dataset.lr_dir={d['lr']} --set dataset.gt_dir={gt} "
            f"--set 'dataset.strip_suffixes={d['strip']}' --results_root {root or RESULTS + '/' + ds} {extra}")

def run_track_a(ds, extra=""):
    if finished(ds, TRACK_A):
        print(f"{ds}/{TRACK_A}: already completed, skipped"); return
    sh(f"bash experiments/scripts/run_inference.sh baseline {_sets(ds, extra)}")

def run_track_b(ds, seeds=None, variants=None, extra=""):
    todo = [n for n in track_b(seeds, variants) if not finished(ds, n)]
    if not todo:
        print(f"{ds}: track B already completed for {variants or VARIANTS}, seeds {seeds or SEEDS}"); return
    # --only names exact runs, so the sweep's default variant filter does not apply
    sh(f"bash experiments/scripts/run_inference.sh sweep --only {' '.join(todo)} {_sets(ds, extra)}")

def evaluate(ds, experiments, data=None, root=None):
    d = data or DATASETS[ds]; root = root or f"{RESULTS}/{ds}"
    for exp in experiments:
        if not finished(ds, exp, root):
            raise RuntimeError(f"{ds}/{exp} has not been run")
        out = f"{root}/{exp}/eval"
        kinds = ["noref", "lrc"] + (["paired"] if d["gt"] else [])
        sums = [pathlib.Path(out, f"{k}_summary.json") for k in kinds]
        if all(f.exists() and not json.loads(f.read_text())["problems"] for f in sums):
            print(f"{ds}/{exp}: already evaluated without problems, skipped"); continue
        cmd = (f"{PY} experiments/evaluation/evaluate.py --sr_dir {root}/{exp}/sr/sample00 --output_dir {out} "
               f"--config experiments/configs/baseline.yaml --noref --lrc --lr_dir {d['lr']} --scale 4 --overwrite")
        if d["gt"]:
            cmd += f" --paired --gt_dir {d['gt']}" + "".join(f" --strip_suffix {s}" for s in d["strip"])
        tag = pathlib.Path(root).name
        sh(cmd + f" 2>&1 | tee {RESULTS}/logs/eval_{tag}_{exp}.log; exit ${{PIPESTATUS[0]}}")

def compare(ds, seeds=None, variants=None):
    root = f"{RESULTS}/{ds}"
    sh(f"{PY} experiments/tools/compare_results.py reproduction --results_root {root} --dataset {ds} --experiment {TRACK_A}")
    sh(f"{PY} experiments/tools/compare_results.py sampling --results_root {root} --dataset {ds} "
       f"--experiments {' '.join(track_b(seeds, variants))} --reference_steps 50")

def run_dataset(ds, seeds=None, variants=None, extra="", reproduce=True):
    if reproduce:
        run_track_a(ds, extra)
    run_track_b(ds, seeds, variants, extra)
    evaluate(ds, ([TRACK_A] if reproduce else []) + track_b(seeds, variants))
    if reproduce:
        compare(ds, seeds, variants)
    else:
        sh(f"{PY} experiments/tools/compare_results.py sampling --results_root {RESULTS}/{ds} --dataset {ds} "
           f"--experiments {' '.join(track_b(seeds, variants))} --reference_steps 50")

## 10. RealSR (tracks A and B, evaluation, comparisons)

Order inside: track A 50 steps, then track B (every variant in `VARIANTS` × 50 / 30 / 20 × `SEEDS`), then paired,
no-reference and LR-consistency evaluation, then `comparison/reproduction.md` and
`comparison/sampling_comparison.md`. Suggested order on a fresh session: first
`run_dataset("RealSR", variants=["original"])` (upstream behaviour, 3 runs), then the full call below.

In [ ]:
run_dataset("RealSR")

## 11. DRealSR

In [ ]:
run_dataset("DRealSR")

## 12. RealLR200 (no GT: no-reference metrics only)

In [ ]:
run_dataset("RealLR200")

## 13. Add seeds or variants later (optional)

Set `SEEDS = [42, 123, 2026]` in section 2 (or pass `seeds=` / `variants=` here) and re-run. Completed runs are
skipped and the sampling comparison is regenerated with everything available.

In [ ]:
# run_dataset("RealSR", seeds=[42, 123, 2026])
# run_dataset("RealSR", variants=["lr_schedule_late"], reproduce=False)   # direction ablation for component B

## 14. Optional: semantic tag agreement and the confidence check for component A (GT-paired datasets)

HR tags are RAM predictions, not ground truth, and DAPE is distilled from RAM; compare variants only.
`tools/analyze_tag_confidence.py` then checks whether DAPE's confidence score `c_i` actually ranks tags by
agreement with RAM-on-HR (the falsification test for component A; see `cfa_seesr/README.md` §7).

In [ ]:
for ds in ("RealSR", "DRealSR"):
    for exp in [TRACK_A] + track_b():
        out = f"{RESULTS}/{ds}/{exp}/eval"
        if pathlib.Path(out, "semantic_summary.json").exists():
            continue
        sh(f"{PY} experiments/evaluation/semantic_tags.py --sr_dir {RESULTS}/{ds}/{exp}/sr/sample00 "
           f"--gt_dir {DATASETS[ds]['gt']} --output_dir {out}" + "".join(f" --strip_suffix {s}" for s in DATASETS[ds]['strip']))
    for exp in track_b(variants=["semantic_conf"]):
        run = f"{RESULTS}/{ds}/{exp}"
        if pathlib.Path(run, "semantic_confidence.jsonl").exists() and pathlib.Path(run, "eval", "semantic_per_image.csv").exists():
            sh(f"{PY} experiments/tools/analyze_tag_confidence.py --run {run} "
               f"--reference {run}/eval/semantic_per_image.csv --min_confidence 0.7310585786300049")

## 15. SSR screening (only if Gate 0 = GO)

20 RealSR + 20 DRealSR images (chosen by filename hash; symlinked, not copied), seed 42, 20 and 50 steps,
five variants: original, semantic_conf, spatial_sem, semantic_spatial, spatial_shuffled. Results go to
`<RESULTS>/screening/<ds>/`, separate from the main results. `tools/ssr_screening.py` then decides:
**CONTINUE** only if SSR improves LPIPS and DISTS consistently (both datasets, both step counts, ≥ 60 % of images)
and the shuffled control does not reproduce the improvement; otherwise **STOP**.

In [ ]:
import hashlib

def screening_subset(ds, n=SCREEN_N):
    """Symlink n paired images (LR + matching GT) into <RESULTS>/screening/<ds>_subset; images are not modified."""
    d = DATASETS[ds]; dst = pathlib.Path(RESULTS, "screening", f"{ds}_subset")
    exts = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp")
    def key(stem):
        for sfx in d["strip"]:
            if stem.endswith(sfx): stem = stem[: -len(sfx)]
        return stem
    gt = {key(p.stem): p for p in pathlib.Path(d["gt"]).iterdir() if p.suffix.lower() in exts}
    lr = sorted((p for p in pathlib.Path(d["lr"]).iterdir() if p.suffix.lower() in exts and key(p.stem) in gt),
                key=lambda p: hashlib.sha256(f"{ds}/{p.name}".encode()).hexdigest())[:n]
    for sub, files in (("lr", lr), ("gt", [gt[key(p.stem)] for p in lr])):
        (dst / sub).mkdir(parents=True, exist_ok=True)
        for p in files:
            link = dst / sub / p.name
            if not link.exists(): link.symlink_to(p)
    return {"lr": str(dst / "lr"), "gt": str(dst / "gt"), "strip": d["strip"]}

SCREEN = "NOT RUN"
if GATE0 != "GO":
    print(f"Gate 0 = {GATE0}: SSR screening skipped; the main evaluation uses the existing two components only.")
else:
    roots = []
    for ds in ("RealSR", "DRealSR"):
        data, root = screening_subset(ds), f"{RESULTS}/screening/{ds}"
        names = [f"{v}_ddpm{st}_s{SCREEN_SEED}" for v in SCREEN_VARIANTS for st in SCREEN_STEPS]
        todo = [n for n in names if not finished(ds, n, root)]
        if todo:
            sh(f"bash experiments/scripts/run_inference.sh sweep --only {' '.join(todo)} {_sets(ds, data=data, root=root)}")
        evaluate(ds, names, data=data, root=root)
        roots.append(f"--results_root {ds}={root}")
    sh(f"{PY} experiments/tools/ssr_screening.py {' '.join(roots)} --steps {' '.join(map(str, SCREEN_STEPS))} "
       f"--seed {SCREEN_SEED} --output {RESULTS}/screening/ssr_screening.json || true")
    SCREEN = json.loads(pathlib.Path(RESULTS, "screening", "ssr_screening.json").read_text())["decision"]
print("SSR SCREENING DECISION:", SCREEN)

## 16. SSR full evaluation (only if screening = CONTINUE)

The SSR variants on the full datasets, in the same results folders as the main track-B runs (same images, seeds,
steps and settings), so `compare_results.py sampling` compares them with the existing variants directly.

In [ ]:
if SCREEN != "CONTINUE":
    print(f"SSR screening = {SCREEN}: full SSR evaluation skipped.")
else:
    for ds in ("RealSR", "DRealSR", "RealLR200"):
        run_dataset(ds, variants=VARIANTS + SSR_VARIANTS, reproduce=False)

## 17. What was saved

In [ ]:
for f in sorted(pathlib.Path(RESULTS).rglob("*")):
    if f.suffix in (".json", ".csv", ".md", ".log", ".txt") and "sr" not in f.relative_to(RESULTS).parts:
        print(f.relative_to(RESULTS))
for ds in DATASETS:   # per-dataset comparisons
    for name in ("reproduction.md", "sampling_comparison.md"):
        f = pathlib.Path(RESULTS, ds, "comparison", name)
        if f.exists():
            print(f.read_text())
# Flush pending Drive writes before ending the session.
from google.colab import drive
drive.flush_and_unmount()

## Notes

* **Runtime and VRAM:** every run records per-image prompt and diffusion time (CUDA-synchronised, first image
  reported separately as warm-up) and peak allocated/reserved VRAM; the sampling comparison tabulates the
  latency change against the reference. Timings are only comparable within one session on one GPU type.
* **SSR:** never in `default_variants`. Gate 0 needs only DAPE (minutes); the screening is 2 datasets × 5 variants ×
  2 step counts on 20 images each. Do not run SSR variants on the full sets unless both gates pass.
* **Budget:** the default grid is 4 variants × 3 step counts × 1 seed = 12 runs per dataset; three seeds triple it.
* **A100:** no changes needed. Do not change tile sizes, precision or `process_size` between compared runs.
* **xformers:** if section 5 reports it unusable, pass `extra="--set runtime.enable_xformers=auto"` to
  `run_dataset` for **every** run so all experiments use the same attention backend (the sampling comparison
  rejects mixed backends).
* **Reproducibility:** seeded but not bitwise deterministic (cuDNN/xformers kernels, fp16).
* **Disconnects:** re-run cells; completed experiments are skipped, interrupted ones must be inspected and rerun with `--overwrite`.